# Step 3: leak experiment on Kaggle

Measures whether input normalisation changes accuracy on partial drawings. Results are on validation data;
the test split is not opened. The decision rule is fixed in `configs/leak_v1.yaml`.

**How to run:** attach your Kaggle Dataset, turn **Internet on**, choose **GPU T4 x2**, then use
**Save Version → Save & Run All**. This runs in the background, so you can close the tab, and it keeps the
output files. `MAX_MINUTES` below caps the run: finished jobs stay valid if it is reached.


In [ ]:
import os, shutil, subprocess
from pathlib import Path

REPO_URL = "https://github.com/Trung4n/AI-dictionary.git"
BRANCH = "pipeline-step1"
REPO = Path("/kaggle/working/repo")

token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    print("No GITHUB_TOKEN secret: cloning without one (works for a public repository).")

url = REPO_URL.replace("https://", f"https://x-access-token:{token}@") if token else REPO_URL
os.chdir("/kaggle/working")  # a re-run may still be inside the folder deleted next
if REPO.exists():
    shutil.rmtree(REPO)
done = subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, str(REPO)], capture_output=True, text=True)
if done.returncode != 0:
    message = done.stderr.replace(token, "***") if token else done.stderr
    raise RuntimeError(f"clone failed:\n{message}")
# Keep the token out of files that Kaggle may save with the notebook output.
subprocess.run(["git", "-C", str(REPO), "remote", "set-url", "origin", REPO_URL], check=True)
print(subprocess.run(["git", "-C", str(REPO), "log", "-1", "--oneline"], capture_output=True, text=True).stdout)


In [ ]:
%cd /kaggle/working/repo
!pip install -q pytest
tests = subprocess.run("PYTHONPATH=src python -m pytest tests/test_exp_setup.py tests/test_baseline.py tests/test_step3.py -q", shell=True, capture_output=True, text=True)
print(tests.stdout[-1500:])
if tests.returncode != 0:
    raise RuntimeError("tests failed: stopping before using GPU quota")


In [ ]:
MAX_MINUTES = 200
!PYTHONPATH=src python -m quickdraw_exp.experiments --matrix configs/leak_v1.yaml --max-minutes {MAX_MINUTES}


In [ ]:
import json

top = Path("/kaggle/working/runs/leak_v1")
result = json.loads((top / "results.json").read_text())
print("COMPLETE" if result["complete"] else "INCOMPLETE, missing: " + ", ".join(result["missing"]))
print(json.dumps(result["decision"], indent=2))
print(json.dumps(result["speed"], indent=2))
for name, row in result["table"].items():
    print(name, "early top-1", round(row["early_top1"]["mean"], 4), [round(x, 4) for x in row["early_top1"]["per_seed"]])
    for frac, scores in row["by_frac"].items():
        print(f"  {frac:>5}  top1 {scores['top1']['mean']:.4f} +- {scores['top1']['std']:.4f}  macro_f1 {scores['macro_f1']['mean']:.4f}  ece {scores['ece']['mean']:.4f}")
for name in result["missing"]:
    log = top / f"{name}.log"
    if log.exists():
        print(f"--- {name} ---")
        print("\n".join(log.read_text().splitlines()[-15:]))
